In [60]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path().resolve().parent
DATA_DIR = PROJECT_ROOT / "data"
EIA_DIR = DATA_DIR / "us_eia"

t1_path = EIA_DIR / "Table1_1.xlsx"
raw = pd.read_excel(t1_path, header=None)

In [61]:
def flatten_header(raw_df: pd.DataFrame, top_row: int = 10, bottom_row: int = 11) -> list[str]:
    header_top = raw_df.iloc[top_row].fillna("")
    header_bottom = raw_df.iloc[bottom_row].fillna("")

    cols = []
    for top, bottom in zip(header_top, header_bottom):
        top = str(top).strip()
        bottom = str(bottom).strip()

        if top and bottom:
            col = f"{top} {bottom}"
        elif top:
            col = top
        else:
            col = bottom

        cols.append(col)

    return cols

In [62]:
def clean_cols(cols: list[str]) -> list[str]:
    result = []
    short_tons_seen = 0
    blank_seen = 0

    for name in cols:
        if name == "NAICS codea":
            result.append("NAICS code")
        elif name == "Totalb (trillion Btu)":
            result.append("Total (trillion Btu)")
        elif name == "electricityc (million kilowatthours)":
            result.append("Net electricity (million kilowatthours)")
        elif name == "fuel oil (million barrels)":
            result.append("Residual fuel oil (million barrels)")
        elif name == "fuel oild (million barrels)":
            result.append("Distillate fuel oil (million barrels)")
        elif name == "gase (billion cubic feet)":
            result.append("Natural gas (billion cubic feet)")
        elif name == "natural gasoline)f (million barrels)":
            result.append("HGL natural gasoline (million barrels)")
        elif name == "(million short tons)":
            short_tons_seen += 1
            if short_tons_seen == 1:
                result.append("Coal (million short tons)")
            else:
                result.append("Coke and breeze (million short tons)")
        elif name == "Hydrogeng (trillion Btu)":
            result.append("Hydrogen (trillion Btu)")
        elif name == "Otherh (trillion Btu)":
            result.append("Other (trillion Btu)")
        elif name == "produced onsitei (trillion Btu)":
            result.append("Net steam and byproduct gases (trillion Btu)")
        elif not name:
            blank_seen += 1
            if blank_seen == 1:
                result.append("Total unit flag (m = million)")
            elif blank_seen == 2:
                result.append("Natural gas unit flag (m = million)")
            elif blank_seen == 3:
                result.append("Other unit flag (m = million)")
            elif blank_seen == 4:
                result.append("Net steam unit flag (m = million)")
            else:
                result.append(f"unnamed_extra_{blank_seen}")
        else:
            result.append(name)

    return result

In [67]:
cols = flatten_header(raw, top_row=10, bottom_row=11)
cleaned_cols = clean_cols(cols)

df1 = raw.iloc[13:].copy()
df1.columns = cleaned_cols
df1 = df1.dropna(axis=0, how="all").reset_index(drop=True)

print(df1.shape)
print(df1.head(10))
df1.columns

(412, 17)
  NAICS code                             Subsector and industry  \
0        NaN                                                NaN   
1        311                                               Food   
2       3112                          Grain and Oilseed Milling   
3     311221          Wet Corn Milling and Starch Manufacturing   
4      31131                                Sugar Manufacturing   
5       3114   Fruit and Vegetable Preserving and Specialty ...   
6       3115                        Dairy Product Manufacturing   
7       3116                 Animal Slaughtering and Processing   
8        312                      Beverage and Tobacco Products   
9       3121                                          Beverages   

  Total (trillion Btu) Total unit flag (m = million)  \
0  Total United States                           NaN   
1                 1235                           NaN   
2                  339                           NaN   
3                  169      

Index(['NAICS code', 'Subsector and industry', 'Total (trillion Btu)',
       'Total unit flag (m = million)',
       'Net electricity (million kilowatthours)',
       'Residual fuel oil (million barrels)',
       'Distillate fuel oil (million barrels)',
       'Natural gas (billion cubic feet)',
       'Natural gas unit flag (m = million)',
       'HGL natural gasoline (million barrels)', 'Coal (million short tons)',
       'Coke and breeze (million short tons)', 'Hydrogen (trillion Btu)',
       'Other unit flag (m = million)', 'Other (trillion Btu)',
       'Net steam unit flag (m = million)',
       'Net steam and byproduct gases (trillion Btu)'],
      dtype='str')

In [71]:
from IPython.display import display, HTML

def show_scrollable(df, height_px=250):
    html = (
        f"<div style='max-height:{height_px}px; overflow:auto; "
        "border:1px solid #444; width:fit-content'>"
        + df.to_html()
        + "</div>"
    )
    display(HTML(html))

show_scrollable(df1, height_px=500)

,NAICS code,Subsector and industry,Total (trillion Btu),Total unit flag (m = million),Net electricity (million kilowatthours),Residual fuel oil (million barrels),Distillate fuel oil (million barrels),Natural gas (billion cubic feet),Natural gas unit flag (m = million),HGL natural gasoline (million barrels),Coal (million short tons),Coke and breeze (million short tons),Hydrogen (trillion Btu),Other unit flag (m = million),Other (trillion Btu),Net steam unit flag (m = million),Net steam and byproduct gases (trillion Btu)
0,NaN,NaN,Total United States,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,311,Food,1235,NaN,79553,*,3,679,NaN,2,4,*,1,NaN,140,m,0
2,3112,Grain and Oilseed Milling,339,NaN,17703,*,*,164,NaN,*,3,0,1,NaN,40,m,0
3,311221,Wet Corn Milling and Starch Manufacturing,169,NaN,7876,0,*,67,NaN,*,2,0,*,NaN,32,NaN,0
4,31131,Sugar Manufacturing,167,NaN,1590,*,*,38,NaN,*,1,*,0,NaN,93,NaN,0
5,3114,Fruit and Vegetable Preserving and Specialty Food,121,NaN,8953,*,*,85,NaN,*,0,0,0,NaN,1,NaN,0
6,3115,Dairy Product Manufacturing,171,NaN,13073,*,*,114,NaN,Q,0,0,0,NaN,Q,NaN,0
7,3116,Animal Slaughtering and Processing,246,NaN,22133,*,1,157,NaN,*,0,0,*,NaN,2,NaN,0
8,312,Beverage and Tobacco Products,265,NaN,14896,0,*,204,NaN,*,*,0,0,NaN,*,NaN,0
9,3121,Beverages,258,NaN,14172,0,*,Q,NaN,*,*,0,0,NaN,*,NaN,0


In [72]:
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent   # already set earlier
DATA_DIR = PROJECT_ROOT / "data"

out_path = DATA_DIR / "table1_1_analysis_ready.csv"
out_path.parent.mkdir(exist_ok=True)

df1.to_csv(out_path, index=False)
out_path

WindowsPath('C:/price_exploration/data/table1_1_analysis_ready.csv')

In [73]:
check_df = pd.read_csv(out_path)
check_df.shape
check_df.head(5)

,NAICS code,Subsector and industry,Total (trillion Btu),Total unit flag (m = million),Net electricity (million kilowatthours),Residual fuel oil (million barrels),Distillate fuel oil (million barrels),Natural gas (billion cubic feet),Natural gas unit flag (m = million),HGL natural gasoline (million barrels),Coal (million short tons),Coke and breeze (million short tons),Hydrogen (trillion Btu),Other unit flag (m = million),Other (trillion Btu),Net steam unit flag (m = million),Net steam and byproduct gases (trillion Btu)
0,NaN,NaN,Total United States,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,311.0,Food,1235,NaN,79553,*,3,679,NaN,2,4,*,1,NaN,140,m,0
2,3112.0,Grain and Oilseed Milling,339,NaN,17703,*,*,164,NaN,*,3,0,1,NaN,40,m,0
3,311221.0,Wet Corn Milling and Starch Manufacturing,169,NaN,7876,0,*,67,NaN,*,2,0,*,NaN,32,NaN,0
4,31131.0,Sugar Manufacturing,167,NaN,1590,*,*,38,NaN,*,1,*,0,NaN,93,NaN,0
